> **데이터셋 안내** — 이 노트북이 참조하는 HF 데이터셋은 공개 배포하지 않는다.
> AI Hub 원본에서 재생성하는 절차는 [docs/data/data-pipeline.md](../docs/data/data-pipeline.md)「가공 데이터셋은 배포하지 않는다 — 재현 경로」에 있다.

In [ ]:
import os
from dotenv import load_dotenv
from pathlib import Path
import json
import numpy as np
import random

load_dotenv()

# 로컬
ROOT = Path(os.environ["DATA_ROOT"])
HF_HOME = ROOT / ".hf_cache"
os.environ["HF_HOME"] = str(HF_HOME)

# 클라우드
# from google.colab import userdata
# os.environ["HF_TOKEN"] = userdata.get("HUGGINGFACEHUB_API_TOKEN")
# os.environ["HF_HOME"] = ".hf_cache"

from huggingface_hub import hf_hub_download
from datasets import load_dataset
from sklearn.metrics import f1_score

In [ ]:
# Config
config = {
    "seed": 42,
    "num_labels": 188,
    "split": "test",
    "tau": 0.5,
    "raw_ds": "ingyoun/patent-clean-text",
    "fields": ["invention_title", "ipc_main", "abstract", "claims"],    # 기록용
    "hf_cache": str(HF_HOME),
    "out_path": ROOT / "output",
}

# 로짓·라벨 연산만 하므로 모델 스펙은 tag/arch만 필요하다(체크포인트·토크나이저는 06_00 참조).
MODELS = [
    {"tag": "kobert-patent-baseline_len512", "arch": "kobert"},
    {"tag": "modernbert-patent-len512", "arch": "modernbert"},
    {"tag": "modernbert-patent-len8192", "arch": "modernbert"},
]

ANCHOR_TAG = "modernbert-patent-len8192"    # 판정 기준 모델(exp1)

In [ ]:
random.seed(config['seed'])
np.random.seed(config['seed'])

In [ ]:
path = hf_hub_download(
    repo_id="ingyoun/patent-clean-text",
    filename="label_mappings.json",
    repo_type="dataset",
)

with open(path, "r", encoding="utf-8") as f:
    label_mapping = json.load(f)

In [ ]:
print(type(label_mapping))
print(label_mapping.keys())

In [ ]:
def show_samples(mappings:dict):
    for i, (k, v) in enumerate(mappings.items()):
        if i <= 5:
            print(f"{k} : {v}")

show_samples(label_mapping["id2mno"])
print("+"*20)
show_samples(label_mapping["mno2lno"])

## Logit

In [ ]:
class LogitLoader:
    def __init__(self, cache_dir: Path, tag: str, split: str):
        self.cache = cache_dir
        self.tag = tag
        self.split = split

    def get(self):
        fp = self.cache / f"logits_{self.tag}_{self.split}.npy"
        if not fp.exists():
            raise FileNotFoundError(f"로짓 캐시 없음: {fp} — 06_00으로 덤프 후 다운로드할 것")
        print(f"[load] {fp.name}")
        return np.load(fp)

In [ ]:
cache_dir = config["out_path"]
logit_dict = {}
for d in MODELS:
    logit_dict[d["tag"]] = LogitLoader(cache_dir=cache_dir, tag=d["tag"], split=config["split"]).get()

prob_dict = {tag: 1.0 / (1.0 + np.exp(-z)) for tag, z in logit_dict.items()}

In [ ]:
prob_dict.keys()

## Hierachy

In [ ]:
class LabelSpace:
    """
    188 Mno 열 ↔ 17 Lno 축
    188-dim 이산 선택(top1 or pred) → [M2L 사영] → 17-dim
    """

    def __init__(self, id2mno: dict, mno2lno: dict, num_labels: int = 188):
        self.C = num_labels
        self.mno_of_col = [id2mno[str(c)] for c in range(self.C)]     # JSON 키는 문자열. (C,)
        self.lno_of_col = [mno2lno[m] for m in self.mno_of_col]       # (C, )
        self.lnos = sorted(set(mno2lno.values()))                     # (17,)
        self.lno_index = {l: i for i, l in enumerate(self.lnos)}      # len = C
        self.L = len(self.lnos)                                       # (17,)  
        self.lno_idx = np.array([self.lno_index[l] for l in self.lno_of_col])     # (C,) 열→Lno
        self.M2L = np.zeros((self.C, self.L), dtype=int)              # (C, L) (188, 17)
        self.M2L[np.arange(self.C), self.lno_idx] = 1                 
        assert self.M2L.sum() == self.C                               # M2L은 행별로 1이 한 개. sum=188

    def to_lno(self, X: np.ndarray) -> np.ndarray:
        """(N, C) Mno 다중핫 → (N, L) Lno 다중핫."""
        return (X.astype(int) @ self.M2L) > 0


LS = LabelSpace(label_mapping["id2mno"], label_mapping["mno2lno"], config["num_labels"])
print(f"C={LS.C}  L={LS.L}  Lno={LS.lnos}")

## 데이터 · 라벨

`ingyoun/patent-clean-text`의 `label_ids`로 188차원 다중핫을 만들고, `length_bin`(`kobert_len` 파생, 고정 축)을 슬라이스 축으로 쓴다.

In [ ]:
ds = load_dataset(config["raw_ds"], split=config["split"])
doc_ids = json.loads((config["out_path"] / f"doc_ids_{config['split']}.json").read_text(encoding="utf-8"))

In [ ]:
# 로짓 행 순서 == 데이터셋 행 순서 (06_00이 DataLoader(shuffle=False)로 보장한 축)
assert ds["document_id"] == doc_ids, "로짓 행 순서와 데이터셋 행 순서가 다르다"
for tag, z in logit_dict.items():
    assert z.shape == (len(ds), config["num_labels"]), tag

N = len(ds)
Y = np.zeros((N, config["num_labels"]), dtype=bool)
for i, ids in enumerate(ds["label_ids"]):
    Y[i, ids] = True

length_bin = np.array(ds["length_bin"])
k_gold = Y.sum(1)                                   # 문서별 정답 라벨 개수. 행방향 합
BINS = ["<=512", "512-1024", "1024-2048", ">2048"]

print(f"N={N:,} || 라벨 개수 : {np.bincount(k_gold)[1:6].tolist()}(k=1 ~ 5) ||  k>=2 비율 {(k_gold >= 2).mean():.2%}")
print("길이 bin " + "  || ".join(f"{b} : {int((length_bin == b).sum()):,}" for b in BINS))

## 앵커(top-1) 오류 분해 — sibling vs cross-Lno

-  sibling 비율이 높다는 것은 모델이 "어느 대분류인지는 대체로 맞히지만, 그 안에서 세부 중분류를 못 가른다"는 뜻이고, 이는 조건부 소프트맥스 $`p(l∣x)⋅p(m∣l,x)`$ 처럼 대분류를 먼저 확정하고 그 안에서 중분류를 고르는 계층 구조가 실제로 도움이 될 여지가 있다는 신호다. 반대로 cross 비율이 높다면 오류가 대분류 경계 자체에서 발생하므로, 계층 구조를 도입해도 1단계(Lno 분류)에서부터 틀려 개선 효과가 제한적이라고 추정할 수 있다.

- sibling 비율은 비정답 클래스에서 균등 추출할 때의 sibling 확률(귀무 기준)과 대비해 읽는다. 계층 확장 여부의 판정은 이 비율이 아니라 「Lno 수준 지표」의 2단계 추정으로 내린다.

In [ ]:
def anchor_decompose(logits: np.ndarray, Y: np.ndarray, ls: LabelSpace):
    """top-1 예측이 정답 집합 밖일 때, 그 Mno의 Lno가 정답 Lno 집합에 있으면 sibling."""
    n = len(Y)
    top1 = logits.argmax(1)         # (N, ), 가장 확신하는 정답
    hit = Y[np.arange(n), top1]     # (n, ), top1의 멀티핫
    err = ~hit
    gold_col = ls.to_lno(Y)[:, ls.lno_idx]        # (N, C) gold_col의 열 c = YL에서, 열 c 자신이 속한 Lno에 해당하는 값
    # gold_col[np.arange(n), top1]  # 문서 i가 1등으로 예측한 Mno의 부모 Lno가, 문서 i의 정답 Lno 집합에 있는가
    sibling = err & gold_col[np.arange(n), top1]  # 대분류는 맞았는데 그 안에서 형제 중분류를 헷갈린 오류
    cross = err & ~gold_col[np.arange(n), top1]   # Mno도 틀렸고, top1의 부모 Lno조차 정답 Lno 집합 밖인 경우. 대분류 자체를 벗어난 오류

    # sibling + cross-Lno == 총 오류 수 · 오류율 == 1 − P@1
    assert int(sibling.sum() + cross.sum()) == int(err.sum())           # mno가 틀린 경우 = lno는 맞춘 경우 + lno도 틀린 경우
    assert abs(float(err.mean()) - (1.0 - float(hit.mean()))) < 1e-12

    # 귀무 기준 — 오답을 비정답 클래스에서 균등 추출할 때 sibling이 될 확률.
    # 대분류당 Mno가 2~20개로 고르지 않아 문서마다 다르므로 오류 문서 평균으로 잡는다.
    # gold_col & ~Y : 정답은 아니지만 뽑히면 sibling으로 카운트 되는 후보 열들
    chance = float(((gold_col & ~Y).sum(1)[err] / (~Y).sum(1)[err]).mean())     # 무작위로 오답을 골랐다면, 그 오답이 정답 대분류 안의 다른 Mno일 확률
    ratio = float(sibling.sum() / err.sum())                                    # 전체 문서 중 sibling 비율

    stats = {
        "p@1": round(float(hit.mean()), 4),
        "n_error": int(err.sum()),
        "error_rate": round(float(err.mean()), 4),
        "sibling": int(sibling.sum()),
        "cross_lno": int(cross.sum()),
        "sibling_ratio": round(ratio, 4),
        "chance_sibling_ratio": round(chance, 4),
        "sibling_enrichment": round(ratio / chance, 2),
    }
    return stats, {"top1": top1, "err": err, "sibling": sibling, "cross": cross}

In [ ]:
anchor, anchor_mask = {}, {}
for d in MODELS:
    anchor[d["tag"]], anchor_mask[d["tag"]] = anchor_decompose(logit_dict[d["tag"]], Y, LS)

header = (f"{'tag':<34}{'P@1':>8}{'오류':>8}"
          f"{'sibling':>14}{'cross-Lno':>16}{'우연':>6}{'배수':>8}")
print(header)
print("-" * len(header))

for tag, r in anchor.items():
    sib = f"{r['sibling']:,} ({r['sibling_ratio']:.1%})"
    cro = f"{r['cross_lno']:,} ({1 - r['sibling_ratio']:.1%})"
    print(f"{tag:<34}{r['p@1']:>8.4f}{r['n_error']:>8,}"
          f"{sib:>16}{cro:>16}"
          f"{r['chance_sibling_ratio']:>8.1%}{r['sibling_enrichment']:>7.1f}x")

## 멀티라벨(τ=0.5) 오류 분해 — FP · FN

In [ ]:
TAU = config["tau"]


def multilabel_decompose(P, Y, ls, tau=TAU):
    """
    FP: 예측한 오답 라벨의 Lno가 정답 Lno 집합에 있는가
    FN: 놓친 정답 라벨의 Lno가 예측 Lno 집합에 있는가(= 대분류는 맞췄으나 중분류를 놓침)
    """
    pred = P >= tau                             # (N, C). 문서 i에서 라벨 C를 예측했는가
    # 문서 i에서 모델이 c(MNO)를 예측했지만 정답이 아니다(FP)
    # 문서 i에서 모델이 c(MNO)가 정답이지만 예측하지 못했다(FN)
    FP, FN = pred & ~Y, Y & ~pred               
    gold_col = ls.to_lno(Y)[:, ls.lno_idx]       # (N, C)
    pred_col = ls.to_lno(pred)[:, ls.lno_idx]    # (N, C)
    # fp_sib = 문서 i에서 모델이 Mno를 잘못 예측했는데(FP), 대분류는 정답 라벨들의 대분류 중 하나와 일치
    # fn_sib = 문서 i에서 모델이 Mno c를 놓쳤는데(FN), 대분류는 모델이 예측한 다른 라벨들의 대분류 중 하나와 일치
    fp_sib, fn_sib = FP & gold_col, FN & pred_col
    return {
        "tau": tau,
        "empty_rate": round(float((pred.sum(1) == 0).mean()), 6),
        "fp": int(FP.sum()),
        "fp_sibling": int(fp_sib.sum()),
        "fp_sibling_ratio": round(float(fp_sib.sum() / FP.sum()), 4),
        "fn": int(FN.sum()),
        "fn_sibling": int(fn_sib.sum()),
        "fn_sibling_ratio": round(float(fn_sib.sum() / FN.sum()), 4),
    }

In [ ]:
multilabel = {d["tag"]: multilabel_decompose(prob_dict[d["tag"]], Y, LS) for d in MODELS}
print(f"{'tag':<34}{'FP':>8}{'FP sib':>16}{'FN':>8}{'FN sib':>16}{'empty':>11}")

for tag, r in multilabel.items():
    print(f"{tag:<34}{r['fp']:>8,}{r['fp_sibling']:>9,} ({r['fp_sibling_ratio']:>5.1%})"
          f"{r['fn']:>8,}{r['fn_sibling']:>9,} ({r['fn_sibling_ratio']:>5.1%}){r['empty_rate']:>9.2%}")

In [ ]:
# empty rate가 기존 SSOT와 일치
for d in MODELS:
    ssot = json.loads((config["out_path"] / f"total_metrics_{d['tag']}.json").read_text(encoding="utf-8"))
    assert abs(multilabel[d["tag"]]["empty_rate"] - ssot["empty_rate_tau_micro"]) < 1e-4, d["tag"]
print("\nverify: empty rate == SSOT (3/3 일치)")

## Lno 수준 지표 · 계층 확장 판정 · 17×17 혼동 행렬

- **판정 기준**: 계층 구조가 이득인지 판정한다. `Mno` 예측을 정답 `Lno` 열로 제한한 **오라클-Lno P@1**(완벽한 Lno 단계를 가정한 상한)에 실제 **Lno 단계 정확도**를 곱한 2단계 추정을 flat P@1과 비교한다. 

- cross-Lno 누수가 특정 대분류 쌍에 집중되면 라벨 경계 혼동으로 볼 수 있지만, 균등하면 분류 과제 자체의 구조적 난이도로 해석할 수 있다.

In [ ]:
def lno_metrics(logits, P, Y, ls, tau=TAU):
    """Mno 예측을 M2L로 사영해 유도한 대분류 성능(별도 Lno 헤드 없음) + 계층 확장 이득 추정."""
    n = len(Y)
    pred = P >= tau
    YL, predL = ls.to_lno(Y), ls.to_lno(pred)
    top1 = logits.argmax(1)
    lno_p1 = float(YL[np.arange(n), ls.lno_idx[top1]].mean())    # Lno 단계 정확도
    flat_p1 = float(Y[np.arange(n), top1].mean())

    # 오라클-Lno: 정답 Lno에 속한 열로만 제한한 뒤 argmax -> 완벽한 Lno 단계를 가정한 상한
    restricted = np.where(YL[:, ls.lno_idx], logits, -np.inf)
    oracle_p1 = float(Y[np.arange(n), restricted.argmax(1)].mean())

    # 2단계 추정 = Lno 단계 정확도 × 조건부 정확도. Lno가 틀리면 회복 불가라는 전제
    two_stage = lno_p1 * oracle_p1        # P(최종 정답)=P(1단계 Lno 적중)×P(2단계 Mno 적중∣1단계 적중)

    return {
        "micro_f1": round(float(f1_score(YL, predL, average="micro", zero_division=0)), 4),
        "macro_f1": round(float(f1_score(YL, predL, average="macro", zero_division=0)), 4),
        "sample_f1": round(float(f1_score(YL, predL, average="samples", zero_division=0)), 4),
        "p@1": round(lno_p1, 4),
        "oracle_lno_p@1": round(oracle_p1, 4),
        "two_stage_p@1_est": round(two_stage, 4),
        "delta_vs_flat": round(two_stage - flat_p1, 4),
    }


def lno_confusion(err, top1, Y, ls):
    """
    앵커 오류 문서의 (정답 Lno × 예측 Lno) 혼동 행렬. 
    정답 Lno가 복수인 문서는 각 정답 Lno에 1씩 계상하므로 행 합이 오류 수보다 클 수 있다
    """
    M = np.zeros((ls.L, ls.L), dtype=int)   # 17 * 17 혼동 행렬
    gold_L = ls.to_lno(Y)                   
    for i in np.where(err)[0]:              # 틀린 문서(err)만 순회
        pl = ls.lno_idx[top1[i]]            # 문서 i의 top1 예측 Mno가 속한 Lno 인덱스
        for gl in np.where(gold_L[i])[0]:   # 정답 Mno의 모든 Lno를 순회(FP, FN)
            M[gl, pl] += 1                  
    return M

In [ ]:
lno = {d["tag"]: lno_metrics(logit_dict[d["tag"]], prob_dict[d["tag"]], Y, LS) for d in MODELS}

confusion = {
    d["tag"]: lno_confusion(anchor_mask[d["tag"]]["err"], anchor_mask[d["tag"]]["top1"], Y, LS)
    for d in MODELS
}

print("[Lno 수준 지표]")
print(f"{'tag':<34}{'micro':>9}{'macro':>9}{'sample':>9}{'P@1':>9}")
for tag, r in lno.items():
    print(f"{tag:<34}{r['micro_f1']:>9.4f}{r['macro_f1']:>9.4f}{r['sample_f1']:>9.4f}{r['p@1']:>9.4f}")

In [ ]:
print("\n[계층 확장 추정 이득]")
print(f"{'tag':<34}{'flat P@1':>10}{'Lno 정확도':>12}{'오라클':>11}{'2단계 추정':>10}{'Δ':>9}")
for d in MODELS:
    tag, r = d["tag"], lno[d["tag"]]
    print(f"{tag:<34}{anchor[tag]['p@1']:>10.4f}{r['p@1']:>14.4f}"
          f"{r['oracle_lno_p@1']:>13.4f}{r['two_stage_p@1_est']:>12.4f}{r['delta_vs_flat']:>+10.4f}")

In [ ]:
delta = lno[ANCHOR_TAG]["delta_vs_flat"]
hierarchy_verdict = {
    "anchor_tag": ANCHOR_TAG,
    "criterion": "2단계(Lno→Mno) 추정 P@1이 flat P@1을 상회하는가 여부",
    "flat_p@1": anchor[ANCHOR_TAG]["p@1"],
    "lno_stage_p@1": lno[ANCHOR_TAG]["p@1"],
    "oracle_lno_p@1": lno[ANCHOR_TAG]["oracle_lno_p@1"],
    "two_stage_p@1_est": lno[ANCHOR_TAG]["two_stage_p@1_est"],
    "delta_vs_flat": delta,
    "decision": "계층 확장 검토" if delta > 0 else "flat 유지",
    "descriptive": {
        "sibling_ratio": anchor[ANCHOR_TAG]["sibling_ratio"],
        "chance_sibling_ratio": anchor[ANCHOR_TAG]["chance_sibling_ratio"],
        "sibling_enrichment": anchor[ANCHOR_TAG]["sibling_enrichment"],
    },
}
print(f"\n판정(기준 {ANCHOR_TAG}): 2단계 추정 {hierarchy_verdict['two_stage_p@1_est']:.4f} vs "
      f"flat {hierarchy_verdict['flat_p@1']:.4f} ({delta:+.4f}) → {hierarchy_verdict['decision']}")

In [ ]:
M = confusion[ANCHOR_TAG]
off = [(M[g, p], LS.lnos[g], LS.lnos[p]) for g in range(LS.L) for p in range(LS.L) if g != p]
off.sort(reverse=True)
print(f"\n[{ANCHOR_TAG}] cross-Lno 누수 상위 10쌍 (정답 → 예측)")
for n, g, p in off[:10]:
    print(f"  {g} → {p}   {n:>4,}")
print(f"  off-diagonal 합계 {sum(x[0] for x in off):,} · 상위 10쌍 점유율 "
      f"{sum(x[0] for x in off[:10]) / max(sum(x[0] for x in off), 1):.1%}")

- 오류의 편중 효과가 적어 국소 처방의 ROI가 낮음
- EA↔EI·LB↔LC의 대칭 혼동이 대분류 경계 자체의 모호성을 시사하며, 이는 계층 구조로 개선되지 않는 오류

## cross-Lno 대칭 쌍 · 국소 처리 상한

위 상위 10쌍은 방향(정답→예측)을 구분한 값이다. 국소 처리(쌍 전문가·클래스 bias 보정)의 여지는 **방향을 합친 무향 쌍**과 그 **대칭성**으로 판단한다 — 두 대분류가 혼동 가능하고 크기가 비슷하면 오류가 양방향으로 흐르는 것이 귀무 기대값이므로, 국소 보정이 통하는 쪽은 비대칭 쌍이다. 이어서 상위 무향 쌍을 완벽히 해소한 상한(앵커 오류를 정답 `Lno` 열로 제한해 재선택)이 계층 확장 판정과 같은 벽에 걸리는지 확인한다.

In [ ]:
def pair_symmetry(M):
    """17×17 혼동 행렬 → 무향 쌍 질량·대칭도, off-diagonal 합계."""
    pairs = []
    for a in range(LS.L):
        for b in range(a + 1, LS.L):
            ab, ba = int(M[a, b]), int(M[b, a])
            tot = ab + ba
            if tot:
                pairs.append({
                    "pair": f"{LS.lnos[a]}<->{LS.lnos[b]}",
                    "ab": ab, "ba": ba, "total": tot,
                    "symmetry": round(min(ab, ba) / max(ab, ba), 3),   # 1.0 = 완전 대칭
                })
    pairs.sort(key=lambda d: -d["total"])
    off_total = int(M.sum() - np.trace(M))
    return pairs, off_total


def pair_oracle_gain(logits, top1, err, pairs, topn):
    """상위 topn 무향 쌍이 걸린 앵커 오류를, 정답 Lno 열로 제한해 재선택한 P@1 이득(상한, pt)."""
    sel = set()
    for p in pairs[:topn]:
        a, b = p["pair"].split("<->")
        ia, ib = LS.lno_index[a], LS.lno_index[b]
        sel |= {(ia, ib), (ib, ia)}
    YL = LS.to_lno(Y)
    fixed = 0
    for i in np.where(err)[0]:
        pl = LS.lno_idx[top1[i]]                 # 예측 top1의 부모 Lno
        gls = np.where(YL[i])[0]                 # 정답 Lno 집합
        if any((gl, pl) in sel for gl in gls):
            zi = np.where(YL[i][LS.lno_idx], logits[i], -np.inf)   # 정답 Lno 열로 제한
            if Y[i, zi.argmax()]:
                fixed += 1
    return {"n_fixed": int(fixed), "p@1_gain_pt": round(100 * fixed / len(Y), 3)}

In [ ]:
pair_analysis = {}
for d in MODELS:
    tag = d["tag"]
    pairs, off_total = pair_symmetry(confusion[tag])
    top1, err = anchor_mask[tag]["top1"], anchor_mask[tag]["err"]
    pair_analysis[tag] = {
        "off_diagonal_total": off_total,
        "top5_pairs": pairs[:5],
        "top5_share": round(sum(p["total"] for p in pairs[:5]) / max(off_total, 1), 4),
        "top10_share": round(sum(p["total"] for p in pairs[:10]) / max(off_total, 1), 4),
        "oracle_gain": {f"top{n}": pair_oracle_gain(logit_dict[tag], top1, err, pairs, n)
                        for n in (1, 5, 10)},
    }

for tag, r in pair_analysis.items():
    print(tag)
    print(f"  off-diag {r['off_diagonal_total']:,} · 무향 상위5 {r['top5_share']:.1%} · 상위10 {r['top10_share']:.1%}")
    print(f"  {'쌍':<12}{'a→b':>7}{'b→a':>7}{'합':>6}{'대칭도':>9}")
    for p in r["top5_pairs"]:
        print(f"  {p['pair']:<12}{p['ab']:>7}{p['ba']:>7}{p['total']:>6}{p['symmetry']:>9.3f}")
    g = r["oracle_gain"]
    print(f"  국소 처리 상한(P@1 이득): 상위1 {g['top1']['p@1_gain_pt']:+.2f}pt · "
          f"상위5 {g['top5']['p@1_gain_pt']:+.2f}pt · 상위10 {g['top10']['p@1_gain_pt']:+.2f}pt\n")

In [ ]:
# verify — 무향 쌍 질량 합 == off-diagonal · 쌍 상한 ⊆ 전체 오라클-Lno 이득
for d in MODELS:
    tag = d["tag"]
    pairs, off_total = pair_symmetry(confusion[tag])
    assert sum(p["total"] for p in pairs) == off_total, tag                    # 무향 쌍이 off-diagonal을 완전 분해
    full_gain = round(100 * (lno[tag]["oracle_lno_p@1"] - anchor[tag]["p@1"]), 3)   # 전체 오라클-Lno 이득(pt)
    top10_gain = pair_analysis[tag]["oracle_gain"]["top10"]["p@1_gain_pt"]
    assert top10_gain <= full_gain + 1e-9, (tag, top10_gain, full_gain)        # 쌍 상한은 전체의 부분집합
print("verify(대칭 쌍) pass — 무향 분해·부분집합 관계 성립")

- **대칭성 자체는 레버가 아니다.** 상위 쌍 대칭도가 0.65~0.96으로 전부 대칭이라 클래스 bias 보정이 통할 비대칭 쌍이 없다. "혼동 가능한 쌍이 있다"는 sibling enrichment(우연 대비 ~5배)가 이미 담은 사실이다.
- **쌍 단위 국소 처리 = 닫힌 계층 확장 갈래의 부분집합.** 상위 5쌍을 완벽히 해소해도 상한이 +1.35pt(exp1)이며, 어떤 쌍 전문가도 "이 문서가 어느 대분류인가"를 먼저 판정하는 게이트를 요구해 `Lno` 단계와 같은 비용을 문다. 판정은 위 2단계 추정과 동일(flat 유지).

## 라벨 개수 bin — 단일(k=1) vs 다라벨(k≥2)

`modernbert-comparison.md` label-cardinality

In [ ]:
def r_precision(P, Y):
    """문서별 상위 |정답| 예측의 정밀도"""
    order = np.argsort(-P, axis=1)
    k = Y.sum(1)
    hits = np.array([Y[i, order[i, :k[i]]].sum() for i in range(len(Y))], dtype=float)
    return hits / np.maximum(k, 1)


def count_bin_decompose(P, Y, tau=TAU):
    pred = P >= tau
    k = Y.sum(1)
    rp = r_precision(P, Y)
    inter = (pred & Y).sum(1)
    sample_f1 = 2 * inter / np.maximum(pred.sum(1) + k, 1)
    out = {}
    for name, m in [("k=1", k == 1), ("k>=2", k >= 2)]:
        fp, fn = int((pred & ~Y)[m].sum()), int((Y & ~pred)[m].sum())
        out[name] = {
            "n": int(m.sum()),
            "micro_f1": round(float(f1_score(Y[m], pred[m], average="micro", zero_division=0)), 4),
            "sample_f1": round(float(sample_f1[m].mean()), 4),
            "r_precision": round(float(rp[m].mean()), 4),
            "fp": fp,
            "fn": fn,
            "fp_fn_ratio": round(fp / max(fn, 1), 4),
        }
    return out

In [ ]:
count_bin = {d["tag"]: count_bin_decompose(prob_dict[d["tag"]], Y) for d in MODELS}

print(f"{'tag':<34}{'bin':>4}{'n':>8}{'micro':>9}{'sample':>9}{'R-Prec':>9}{'FP':>8}{'FN':>8}{'FP:FN':>8}")
for tag, r in count_bin.items():
    for name in ["k=1", "k>=2"]:
        v = r[name]
        print(f"{tag:<34}{name:>4}{v['n']:>8,}{v['micro_f1']:>9.4f}{v['sample_f1']:>9.4f}"
              f"{v['r_precision']:>9.4f}{v['fp']:>8,}{v['fn']:>8,}{v['fp_fn_ratio']:>8.2f}")
    gap = r["k=1"]["sample_f1"] - r["k>=2"]["sample_f1"]
    print(f"{'':<34}{'[sample f1(k=1) - sample f1(k>=2) : ':>7}{'':>8}{'':>9}{gap:>9.4f}]\n")

- 멀티 라벨의 성능이 모든 모델에서 저하. 딘일 라벨 샘플이 다수라 모델의 전체 성능이 높게 유지됨. 멀티 라벨의 성능이 단일 라벨에 가려져 있음

## 라벨 개수 bin — 카디널리티 헤드룸 (주 지표 환산)

위 bin 분해는 sample-F1·R-Precision 기준이다. 주 지표는 micro-F1이므로 k≥2의 무게를 **문서 비율이 아니라 양성 라벨 인스턴스 비율**로 환산하고, 결손이 표현(랭킹)에서 오는지 결정 규칙(카디널리티)에서 오는지 가른다 — k≥2 문서에만 **오라클 카디널리티**(정답 개수 `k`를 알고 상위 `k`개 선택, 랭킹 불변)를 적용한 micro-F1이 도달 불가 상한이다.

In [ ]:
def cardinality_analysis(logits, P, Y, tau=TAU):
    """k≥2 결손을 주 지표(micro)로 환산 — 오라클 카디널리티 상한·과소예측 진단."""
    k = Y.sum(1)
    pred = P >= tau
    m2 = k >= 2
    cf = pred.copy()                              # k≥2 문서만 상위 k개로 교체(랭킹 불변)
    for i in np.where(m2)[0]:
        row = np.zeros(pred.shape[1], dtype=bool)
        row[np.argpartition(-logits[i], k[i])[:k[i]]] = True
        cf[i] = row
    micro = float(f1_score(Y, pred, average="micro", zero_division=0))
    micro_oracle = float(f1_score(Y, cf, average="micro", zero_division=0))
    k_pred = pred.sum(1)
    return {
        "pos_instances_k1": int(Y[k == 1].sum()),
        "pos_instances_k>=2": int(Y[m2].sum()),
        "pos_share_k>=2": round(float(Y[m2].sum() / Y.sum()), 4),
        "micro": round(micro, 4),
        "micro_oracle_k_on_multi": round(micro_oracle, 4),
        "oracle_k_gain_pt": round(100 * (micro_oracle - micro), 3),
        "under_predict_rate_k>=2": round(float((k_pred[m2] < k[m2]).mean()), 4),
        "mean_k_pred_k>=2": round(float(k_pred[m2].mean()), 3),
        "mean_k_gold_k>=2": round(float(k[m2].mean()), 3),
        "mean_k_pred_k1": round(float(k_pred[k == 1].mean()), 3),
    }

In [ ]:
cardinality = {d["tag"]: cardinality_analysis(logit_dict[d["tag"]], prob_dict[d["tag"]], Y)
               for d in MODELS}

print(f"양성 라벨 인스턴스 {int(Y.sum()):,} · k≥2 점유율 {cardinality[MODELS[0]['tag']]['pos_share_k>=2']:.1%}\n")
print(f"{'tag':<34}{'micro':>9}{'+오라클k':>11}{'이득':>9}{'k≥2과소예측':>13}{'예측/정답':>12}")
for tag, r in cardinality.items():
    kp = f"{r['mean_k_pred_k>=2']:.2f}/{r['mean_k_gold_k>=2']:.2f}"
    print(f"{tag:<34}{r['micro']:>9.4f}{r['micro_oracle_k_on_multi']:>12.4f}"
          f"{r['oracle_k_gain_pt']:>+12.2f}{r['under_predict_rate_k>=2']:>15.1%}{kp:>16}")

In [ ]:
# verify — micro 재계산 == SSOT(total_metrics.keep.micro) · 오라클-k sample-F1 == R-Precision(k≥2)
for d in MODELS:
    tag = d["tag"]
    ssot = json.loads((config["out_path"] / f"total_metrics_{tag}.json").read_text(encoding="utf-8"))
    assert abs(cardinality[tag]["micro"] - round(ssot["multilabel_f1"]["keep"]["micro"], 4)) < 1e-4, tag
    k = Y.sum(1); m2 = k >= 2; logits = logit_dict[tag]
    cf = np.zeros((int(m2.sum()), Y.shape[1]), dtype=bool)          # k개 정확히 선택 → sample-F1 = R-Precision
    for j, i in enumerate(np.where(m2)[0]):
        cf[j, np.argpartition(-logits[i], k[i])[:k[i]]] = True
    sf1 = round(float(f1_score(Y[m2], cf, average="samples", zero_division=0)), 4)
    assert abs(sf1 - count_bin[tag]["k>=2"]["r_precision"]) < 1e-4, (tag, sf1, count_bin[tag]["k>=2"]["r_precision"])
print("verify(카디널리티) pass — micro SSOT 일치 · 오라클-k sample-F1 == R-Precision(k≥2)")

- **k≥2는 문서 15.0%지만 양성 라벨 인스턴스의 ~29.4%**를 갖는다 — micro 기준에선 이 비율로 읽는다.
- **결손의 대부분은 표현이 아니라 결정 규칙.** 오라클-k는 랭킹을 바꾸지 않는데도 exp1 micro를 +1.63pt 올려 exp1이 KoBERT 재현선 대비 번 전체 개선(+1.83pt)에 맞먹는다. 3모델 공통(+1.43~+1.63pt).
- **기제는 다라벨 문서 한정 과소예측**(k≥2 과소예측 ~44%, 평균 예측<정답). k=1은 오히려 과대예측이라 전역 캘리브레이션이 아니다.
- **임계값으로 닿지 않는다**(단일 전역 τ는 문서별 카디널리티를 표현 못 함 — `06_02` B). 회수 경로는 손실 함수(`../NEXT_SESSION.md` 2단계). 오라클-k는 상한이지 목표치가 아니다.

## 길이 bin × 오류 유형

B3(>2048)에서 macro가 급락하는 원인이 cross-Lno 증가(표현력)인지 FN 증가(임계값)인지 가른다. 처방이 갈리는 지점.

In [ ]:
def bin_error_types(P, Y, err, sibling, cross, tau=TAU):
    pred = P >= tau
    FP, FN = pred & ~Y, Y & ~pred
    out = {}
    for b in BINS:
        m = length_bin == b
        n_err = int(err[m].sum())
        out[b] = {
            "n": int(m.sum()),
            "anchor_error_rate": round(float(err[m].mean()), 4),
            "sibling": int(sibling[m].sum()),
            "cross_lno": int(cross[m].sum()),
            "sibling_ratio": round(float(sibling[m].sum() / max(n_err, 1)), 4),
            "fp_per_doc": round(float(FP[m].sum() / m.sum()), 4),
            "fn_per_doc": round(float(FN[m].sum() / m.sum()), 4),
        }
    return out

In [ ]:
bin_error = {
    d["tag"]: bin_error_types(prob_dict[d["tag"]], Y, anchor_mask[d["tag"]]["err"],
                              anchor_mask[d["tag"]]["sibling"], anchor_mask[d["tag"]]["cross"])
    for d in MODELS
}

for tag, r in bin_error.items():
    print(tag)
    print(f"  {'bin':<12}{'n':>7}{'오류율':>8}{'sibling비':>10}{'FP/문서':>9}{'FN/문서':>8}")
    for b in BINS:
        v = r[b]
        print(f"  {b:<12}{v['n']:>7,}{v['anchor_error_rate']:>9.2%}{v['sibling_ratio']:>11.1%}"
              f"{v['fp_per_doc']:>10.3f}{v['fn_per_doc']:>10.3f}")
    print()

## 3모델 오류 차집합 — 모델 성분 · 창 성분

`modernbert-comparison.md` 「길이 vs 모델 분해」를 오류 수준에서 재현한다. 창 512를 고정한 쌍(KoBERT→A.X@512)이 모델 성분, A.X를 고정한 쌍(A.X@512→A.X@8192)이 창 성분이다. 창 성분의 교정률이 장문 bin으로 갈수록 높아지면 `06_03`이 실측한 커버리지 기제(절단 문서 +10.0%)와 정합한다.

In [ ]:
# (base, target, 성분) — 한 쌍만 보면 모델 성분과 창 성분이 섞이므로 세 쌍을 함께 낸다.
PAIRS = [
    ("kobert-patent-baseline_len512", "modernbert-patent-len512", "model"),    # 창 512 고정 → 모델 성분
    ("modernbert-patent-len512", "modernbert-patent-len8192", "window"),       # A.X 고정 → 창 성분
    ("kobert-patent-baseline_len512", "modernbert-patent-len8192", "total"),   # 합산
]


def diff_profile(mask, err_owner):
    """차집합 문서의 오류 유형(err_owner 기준)·길이 bin 분포."""
    sib, cro = anchor_mask[err_owner]["sibling"], anchor_mask[err_owner]["cross"]
    return {
        "n": int(mask.sum()),
        "sibling": int((mask & sib).sum()),
        "cross_lno": int((mask & cro).sum()),
        "by_length_bin": {b: int((mask & (length_bin == b)).sum()) for b in BINS},
    }


def compare(base, target, component):
    base_err, target_err = anchor_mask[base]["err"], anchor_mask[target]["err"]
    fixed = base_err & ~target_err      # base가 틀리고 target이 맞춘 문서
    broken = ~base_err & target_err     # 그 반대
    return {
        "component": component,
        "base": base,
        "target": target,
        "fixed": diff_profile(fixed, base),
        "broken": diff_profile(broken, target),
        "net_gain": int(fixed.sum() - broken.sum()),
        # bin별 순이득 — 창 성분이 절단 없는 B0에서 0 부근인지가 커버리지 기제의 확증점
        "net_by_bin": {
            b: int((fixed & (length_bin == b)).sum() - (broken & (length_bin == b)).sum())
            for b in BINS
        },
        # bin별 교정률 = fixed / base 오류
        "fix_rate_by_bin": {
            b: round(float((fixed & (length_bin == b)).sum()
                           / max(int((base_err & (length_bin == b)).sum()), 1)), 4)
            for b in BINS
        },
    }


hard_core = np.logical_and.reduce([anchor_mask[d["tag"]]["err"] for d in MODELS])

cross_model = {
    "pairs": [compare(*p) for p in PAIRS],
    "hard_core": {
        "n": int(hard_core.sum()),
        "by_length_bin": {b: int((hard_core & (length_bin == b)).sum()) for b in BINS},
    },
}

for r in cross_model["pairs"]:
    print(f"[{r['component']}] {r['base']} → {r['target']}")
    print(f"  {'':<10}{'n':>7}{'sibling':>10}{'cross':>8}   " + "".join(f"{b:>12}" for b in BINS))
    for name in ["fixed", "broken"]:
        v = r[name]
        print(f"  {name:<10}{v['n']:>7,}{v['sibling']:>10,}{v['cross_lno']:>8,}   "
              + "".join(f"{v['by_length_bin'][b]:>12,}" for b in BINS))
    print(f"  {'순이득':<8}{r['net_gain']:>+7,}{'':>18}   "
          + "".join(f"{r['net_by_bin'][b]:>+12,}" for b in BINS))
    print(f"  {'교정률':<8}{'':>7}{'':>18}   "
          + "".join(f"{r['fix_rate_by_bin'][b]:>12.1%}" for b in BINS) + "\n")

print(f"3모델 공통 오류(hard core) {cross_model['hard_core']['n']:,}건")
print("  bin별 " + "  ".join(f"{b} {cross_model['hard_core']['by_length_bin'][b]:,}" for b in BINS))

## 저장

In [ ]:
meta = {
    "split": config["split"],
    "n_docs": int(N),
    "tau": TAU,
    "num_labels": config["num_labels"],
    "num_lno": LS.L,
    "fields": config["fields"],
    "raw_ds": config["raw_ds"],
}

for d in MODELS:
    tag = d["tag"]
    result = {
        **meta,
        "tag": tag,
        "arch": d["arch"],
        "anchor_error": anchor[tag],
        "multilabel_error": multilabel[tag],
        "lno_metrics": lno[tag],
        "lno_confusion": {"lnos": LS.lnos, "matrix": confusion[tag].tolist()},
        "label_count_bins": count_bin[tag],
        "cardinality": cardinality[tag],
        "pair_analysis": pair_analysis[tag],
        "length_bin_error": bin_error[tag],
    }
    fp = config["out_path"] / f"error_analysis_{tag}.json"
    fp.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"saved: {fp}")

fp = config["out_path"] / "error_analysis_cross_model.json"
fp.write_text(json.dumps({**meta, **cross_model, "hierarchy_verdict": hierarchy_verdict},
                         ensure_ascii=False, indent=2), encoding="utf-8")
print(f"saved: {fp}")